# Structural Features in VEP-nAChR2 — Complete Guide

## What they are, why they matter, how we compute them, and what we fixed

**Date:** 2026-08-12 | **Context:** Run III preparation — DSSP/ASA fix

---
## 1. Overview: Two Structural Feature Groups

VEP-nAChR2 has **two** structural feature groups, totaling **14 features** (out of 66 total):

| Group | Extractor Class | Features | Requires |
|-------|----------------|----------|----------|
| **structural_core** (7) | `StructuralExtractor` | RSA, Cβ-density, B-factor, DSSP helix/sheet/coil, is_unmappable | PDB + Reference seq |
| **structural_nachr** (7) | `StructuralNachrExtractor` | TM helix, TM depth, pore distance, ligand proximity, interface proximity, interface contacts, subunit burial | PDB + Reference seq |

Plus one conditional group:

| Group | Extractor Class | Features | Active for |
|-------|----------------|----------|------------|
| **conformational** (5) | `ConformationalExtractor` | ΔRSA, ΔB-factor, Δinterface, CA RMSD, pore radius change | CHRNA7 only |

---
## 2. structural_core — The VEP-ENaC Clone (7 features)

These are the **universal** PDB-derived features cloned from VEP-ENaC. They work for any protein with a 3D structure.

### 2.1 RSA — Relative Solvent Accessibility

**Full form:** Relative Solvent Accessibility

**What it measures:** How exposed a residue is to solvent (water). Ranges from 0 (completely buried in the protein core) to 1 (fully exposed on the surface).

**Why it matters for GOF/LOF prediction:**
- **Buried residues (RSA < 0.2):** Mutations here typically disrupt protein folding/stability → usually **LOF**
- **Surface residues (RSA > 0.5):** Mutations here affect protein-protein interactions, ligand binding, or post-translational modifications → can be **GOF or LOF**
- **Core mutations are rarely tolerated** — the RSA context tells the model whether a substitution is at a critical structural position

**How it's computed:**
```
RSA = SASA / MAX_ASA[AA]
```
- **SASA** = Solvent Accessible Surface Area (Å²) — computed by rolling a 1.4Å water probe over the protein surface
- **MAX_ASA[AA]** = Maximum possible SASA for that amino acid type (Tien et al., 2013 reference scale)
  - e.g., Glycine max = 104 Å², Tryptophan max = 285 Å²

**Algorithm:** Shrake-Rupley (1973) — the same algorithm used by the DSSP program since v2.0

**Data source:** Computed via `freesasa` Python package from PDB structure files

**The bug we fixed:** Before 2026-08-12, all DSSP files were simplified CIF fallbacks with `ASA=0` for every residue. RSA was computed as `0/MAX_ASA = 0.0` → then filled with `1.0` (the default for unmappable residues). This meant **every variant had RSA=1.0** — complete noise. See Section 5 for details.

### 2.2 Cβ Density — Local Packing Density

**Full form:** C-beta atom density (also called "contact density" or "packing density")

**What it measures:** Number of other residue Cβ atoms (CA for glycine) within a 10 Å radius. Higher values = more tightly packed local environment.

**Why it matters:**
- **High Cβ density:** Residue is in a tightly packed region (protein core, helix bundle). Mutations here are sterically constrained — most substitutions won't fit → typically **LOF**
- **Low Cβ density:** Residue is in a flexible loop or surface region. More tolerant of substitutions.

**How it's computed:**
```python
from scipy.spatial import cKDTree
kdtree = cKDTree(all_cb_coordinates)
neighbors = kdtree.query_ball_point(residue_cb_coord, r=10.0)  # 10 Å radius
cbeta_density = len(neighbors) - 1  # exclude self
```

**Data source:** PDB structure coordinates (Cβ atom positions from all chains). Works correctly regardless of DSSP quality.

### 2.3 B-factor — Thermal Displacement / Flexibility

**Full form:** B-factor (also called Debye-Waller factor, temperature factor, or atomic displacement parameter)

**What it measures:** How much an atom vibrates around its mean position. High B-factor = flexible/disordered region. Low B-factor = rigid/ordered region. Measured in Å².

**Why it matters:**
- **High B-factor regions:** Flexible loops, disordered termini. Mutations here often tolerated.
- **Low B-factor regions:** Rigid core, functional sites. Mutations here often damaging (LOF).
- **B-factor patterns can identify:** active sites (often rigid), allosteric pathways (intermediate flexibility), hinge regions (flexible)

**How it's computed:**
```python
# Average B-factor of all non-hydrogen atoms in the residue
atom_bfs = [atom.bfactor for atom in residue if atom.element != 'H']
b_factor = mean(atom_bfs)
# If residue unmappable, use the chain median B-factor as fallback
```

**Data source:** PDB/mmCIF atom records — every atom in a crystal structure has a B-factor. Works correctly regardless of DSSP quality.

**Limitation for nAChR:** Cryo-EM structures (7EKT, 7KOX, 6CNJ, 6PV7, 9DMG) have less reliable B-factors than X-ray crystallography. AlphaFold structures (CHRNA9, CHRNA10) have **no experimental B-factors** — the AF2 pLDDT scores are stored in the B-factor field as a confidence metric (0-100), not a true thermal parameter. Our pipeline uses them as-is, which means AlphaFold B-factors represent prediction confidence, not flexibility.

### 2.4 DSSP Secondary Structure — 3 one-hot features

**Full form:** Dictionary of Secondary Structure of Proteins

**What it measures:** The local backbone conformation of each residue, classified into 3 categories:
- **Helix (H/G/I):** α-helix, 3₁₀-helix, π-helix
- **Sheet (E/B):** β-strand (extended), β-bridge
- **Coil (everything else):** loops, turns, bends, irregular

**Why it matters:**
- **Helix residues:** Often in transmembrane domains (TM1-TM4 in nAChR). Mutations can disrupt helix packing → **LOF**
- **Sheet residues:** Often in extracellular β-sandwich domain. Ligand binding loops (A, B, C, D, E, F) are between sheet strands
- **Coil/loop residues:** Often at functional sites — binding loops, subunit interfaces. Mutations can alter receptor kinetics → often **GOF**

**How it's computed (in DSSP):**
The real DSSP program (Kabsch & Sander, 1983) assigns secondary structure based on **backbone hydrogen bonding patterns** — it identifies main-chain H-bonds between C=O and N-H groups. This is more accurate than geometric methods.

**Our fallback (in the simplified DSSP):**
We extract helix/sheet annotations from the mmCIF file's `_struct_conf` and `_struct_sheet_range` tables. These are **author-assigned** secondary structure ranges, not computed from H-bonds. They cover ~60-70% of actual secondary structure elements. Residues not in these tables default to coil.

**Representation in features:**
```python
dssp_helix = 1 if ss_code in 'HGI' else 0
dssp_sheet = 1 if ss_code in 'EB'  else 0
dssp_coil  = 1 if neither             else 0
# These always sum to 1 (one-hot encoding)
```

### 2.5 is_unmappable — Structure Coverage Flag

**What it measures:** Binary flag: 1 = this variant position cannot be mapped to any PDB residue, 0 = successfully mapped.

**Why it matters:** Tells the model whether the other structural features are real or fill values. A model can learn to down-weight structural features when `is_unmappable=1`.

**How it's computed:**
```python
# Build alignment between UniProt reference sequence and PDB chain
alignment_map = align_reference_to_pdb_chain(ref_seq, pdb_chain)
# Check if variant position maps to a PDB residue
is_unmappable = 1 if alignment_map.get(position) is None else 0
```

**Coverage by gene (after fix):**

| Gene | PDB | Chain | Mapping Type | Fill Rate |
|------|-----|-------|-------------|-----------|
| CHRNA1 | 9DMG | A | Direct | 2/126 (1.6%) |
| CHRNA7 | 7EKT | A | Direct | 6/207 (2.9%) |
| CHRNA4 | 6CNJ | A | Direct | 41/90 (45.6%) ⚠️ |
| CHRNA3 | 6PV7 | A | Direct | 4/13 (30.8%) |
| CHRNB1 | 9DMG | E | Direct | 2/21 (9.5%) |
| CHRNB2 | 6CNJ | B | Direct | 13/50 (26.0%) |
| CHRNB4 | 6PV7 | B | Direct | 3/53 (5.7%) |
| CHRND | 9DMG | D | Direct | 1/44 (2.3%) |
| CHRNE | 9DMG | B | Direct | 13/71 (18.3%) |
| CHRNA2 | 6CNJ | A | Homology→CHRNA4 | varies |
| CHRNA6 | 6CNJ | A | Homology→CHRNA4 | 9/44 (20.5%) |
| CHRNA5 | 6PV7 | A | Homology→CHRNA3 | varies |
| CHRNB3 | 6PV7 | B | Homology→CHRNB4 | varies |
| CHRNG | 9DMG | B | Homology→CHRNE | 9/9 (100%) ⚠️ |
| CHRNA9 | AF-Q9UGM1 | A | AlphaFold | varies |
| CHRNA10 | AF-Q13002 | A | AlphaFold | varies |

⚠️ **CHRNA4** and **CHRNG** have high fill rates — possible alignment issues needing investigation.

---
## 3. structural_nachr — nAChR-Specific Features (7 features)

These features capture **nAChR-specific biophysics** — things that matter for a pentameric ligand-gated ion channel but wouldn't apply to a generic protein.

### 3.1 TM Helix ID (tm_helix)

**What it is:** Which transmembrane helix (0-4) the residue belongs to. 0 = non-TM (extracellular or intracellular domain).

**Why it matters for nAChR:**
- **TM1:** Outer helix, faces lipid bilayer. Mutations can alter channel assembly.
- **TM2 (M2):** **PORE-LINING HELIX** — the most important TM domain. Lines the ion channel. Mutations directly affect ion conductance, channel gating, and agonist efficacy. Most disease-causing GOF mutations (e.g., autosomal dominant nocturnal frontal lobe epilepsy) are in M2.
- **TM3:** Inner helix, faces TM2 and lipid. Mutations can alter channel kinetics.
- **TM4:** Outermost helix, primarily lipid-facing. Mutations often affect expression/trafficking (LOF).

**How it's computed:** From UniProt-annotated TM boundaries (literature-curated, Unwin 2005). Not PDB-dependent — always available.

### 3.2 TM Depth (tm_depth)

**What it is:** Normalized position within the membrane (0 = extracellular edge, 1 = intracellular edge).

**Why it matters:** The pore gate is at the intracellular end of M2. Mutations near the gate (depth ~0.7-1.0) have stronger effects on channel gating than mutations near the extracellular end.

**How it's computed:** Linear interpolation within the TM helix boundaries. Not PDB-dependent.

### 3.3 Pore Distance (pore_distance)

**What it is:** Distance (Å) from the residue's CA atom to the central pore axis.

**Why it matters:** Residues closer to the pore have larger effects on ion permeation. M2 residues face the pore (distance ~5-10 Å). M4 residues face lipid (distance ~25-30 Å).

**How it's computed:**
```python
# Pore axis = geometric center of M2 helix CA atoms from all 5 subunits
pore_center = mean([m2_ca_atoms for each chain])
pore_distance = distance(residue_ca, pore_center)
```

**⚠️ Known bug:** Uses raw UniProt position as key into PDB residue dict (no alignment). Works correctly for well-annotated structures where PDB numbering = UniProt numbering. Should be fixed by sharing alignment maps from StructuralExtractor.

### 3.4 Ligand Proximity (ligand_proximity)

**What it is:** Distance (Å) to the nearest orthosteric ligand binding site residue.

**Why it matters:** Mutations near the neurotransmitter binding site directly affect agonist binding affinity and efficacy. nAChR has two agonist binding sites per pentamer (at α-δ and α-γ/ε interfaces for muscle type, α-β interfaces for neuronal type).

**Binding site definition:** We use manually curated residue lists from literature for the key binding loops:
- **Loop A** (β9-β10): principal face, contributes to agonist binding
- **Loop B** (β7-β8): principal face
- **Loop C** (β9-β10 C-terminus): principal face — contains the Cys-loop disulfide
- **Loop D** (β2-β3): complementary face
- **Loop E** (β5-β6): complementary face
- **Loop F** (β8-β9): complementary face

Currently curated for CHRNA1 (muscle), CHRNA7, CHRNA4, CHRNA3 only. Other α subunits use homology.

**How it's computed:** Minimum distance from residue CA to any binding site residue CA.

### 3.5 Interface Proximity & Contacts (interface_proximity, interface_contacts)

**What they measure:**
- **interface_proximity:** Minimum CA-CA distance to any residue in a neighboring subunit chain (Å)
- **interface_contacts:** Number of neighboring chain atoms within 5 Å

**Why they matter:** Subunit interfaces are critical for:
- **Assembly:** mutations at interfaces can prevent proper pentamer formation → **LOF**
- **Cooperativity:** agonist binding at one interface affects gating at all subunits
- **Allosteric modulation:** many allosteric sites are at subunit interfaces
- **Uncoupling:** some GOF mutations at interfaces uncouple binding from gating

**How they're computed:** KDTree distance search across all CA atoms in the pentamer.

### 3.6 Subunit Burial (subunit_burial)

**What it is:** Fraction of residue SASA buried by neighboring subunits (SASA_monomer - SASA_pentamer) / SASA_monomer.

**Why it matters:** Identifies residues that become buried upon pentamer assembly. Mutations at assembly interfaces typically affect receptor expression/trafficking (LOF).

**Current status:** Placeholder (returns 0.0). Requires per-chain SASA computation in both monomeric and pentameric contexts.

---
## 4. Conformational Features — Open vs Closed State (5 features, CHRNA7 only)

**What they measure:** Changes in structural properties between the **resting/closed** state (7EKT, antagonist-bound) and **activated/open** state (7KOX, epibatidine+PNU).

**Why they matter:**
A mutation that destabilizes the closed state = **GOF** (receptor activates more easily).
A mutation that destabilizes the open state = **LOF** (receptor can't stay open).

The conformational change features tell the model: "this residue moves X Å during gating; if you mutate it, you'll affect the gating equilibrium by Y."

| Feature | What it measures | Status |
|---------|-----------------|--------|
| delta_rsa | RSA(open) - RSA(closed) | Placeholder — needs mkdssp for ASA |
| delta_bfactor | B-factor(open) - B-factor(closed) | Placeholder |
| delta_interface | Interface proximity change | Placeholder |
| ca_rmsd | Cα displacement between states (Å) | **Working** — computed from 7EKT vs 7KOX alignment |
| pore_radius_change | M2 pore constriction change | Placeholder |

**The CA RMSD is the only active feature** — it captures how much each residue moves during the closed→open transition. Residues with high RMSD are involved in the gating conformational change.

**For non-CHRNA7 genes:** All 5 features = 0.0 (placeholder until other receptor types have open+closed structures).

---
## 5. The DSSP/ASA Bug — What Was Broken and How We Fixed It

### 5.1 The Problem

The DSSP program (`mkdssp`) generates files containing per-residue:
- **Secondary structure** (H/E/C) based on backbone H-bond patterns
- **Solvent Accessible Surface Area** (ASA in Å²)
- **Phi/Psi backbone angles**
- **H-bond energies**

VEP-nAChR2 could not run `mkdssp` because:
1. The user's Python environment is standard `python.org` Python 3.11 (not conda)
2. `mkdssp` is a C++ program, not a Python package — `pip install dssp` doesn't exist
3. The Windows binary from PDB-REDO (v4.4.0) requires **administrator elevation** (error 740) — it's compiled with `requireAdministrator` in its manifest
4. The Windows binary from PDB-REDO (v4.5.8) has **no precompiled assets at all**

### 5.2 The Fallback (What We Had)

`download_pdbs.py` has a 3-tier fallback:
1. Try `mkdssp` (Method 1) → ❌ not installed, can't run
2. Extract SS from mmCIF annotations (Method 2) → ✅ partially works
3. PDBe API for pre-computed DSSP (Method 3) → ❌ network issues

Method 2 produces a **simplified DSSP file** from CIF `_struct_conf` (helices) and `_struct_sheet_range` (sheets) tables. The problem: these tables only contain **secondary structure assignments** — they have **NO ASA values**.

**Result:** Every DSSP file on disk had:
```
==== DSSP file for 9DMG (simplified, SS only) ====
```
Every residue line:
```
    1    1 A XX   H   0    0    0  ...
                              ^^^
                          ASA = 0 for EVERY residue!
```

### 5.3 The Cascade of Failures

```
No mkdssp
    → Simplified DSSP files (ASA=0 everywhere)
        → _compute_rsa(): dssp_data[2] = 0
            → RSA = 0 / MAX_ASA = 0.0
                → fill value (1.0) applied for unmappable
                    → EVERY VARIANT GETS RSA = 1.0
                        → RSA feature = complete noise
                            → structural_core becomes HARMFUL in ablation
```

This is why **Run I** showed structural_core as the **worst** feature group (+0.023 delta F1 — dropping it *improved* performance) and **Run II** confirmed it as noisy (+0.004 average across 10 models).

### 5.4 The Fix (2026-08-12)

**Approach:** Use `freesasa` (Python package, `pip install freesasa`) instead of the `mkdssp` binary.

**Why freesasa is equivalent:**
- Same **Shrake-Rupley algorithm** (1973) as the DSSP program since v2.0
- Same **1.4 Å water probe radius**
- Same per-atom → per-residue SASA summation
- Peer-reviewed, widely used in computational biology
- The only difference: DSSP also computes H-bond patterns for SS assignment (which we get from CIF instead)

**What we did:**

1. **`pip install freesasa`** — one command, works on any OS
2. **Wrote `scripts/regenerate_dssp.py`** — regenerates all 7 DSSP files:
   - Reads PDB structure with freesasa → computes per-residue SASA
   - Parses mmCIF to get secondary structure annotations
   - Writes DSSP-format files with **real ASA values** and **correct AA codes**
   - Format matches Bio.Python's `make_dssp_dict()` parser exactly
3. **Added AlphaFold structures** (AF-Q9UGM1, AF-Q13002) — previously had no DSSP at all

### 5.5 Before vs After

| Metric | Before Fix | After Fix |
|--------|-----------|----------|
| Unique RSA values | 1 (all = 1.0) | 368 distinct values |
| Variants with real RSA | 0 / 797 (0%) | 693 / 797 (87%) |
| RSA distribution | Flat at 1.0 | Buried=47%, Partial=28%, Exposed=25% |
| PDBs with DSSP | 5 (all broken) | 7 (all working) |
| ASA>0 residues | 0 (all DSSP files) | 88-97% per PDB |
| Bio.Python DSSP parse | Failed (wrong column format) | All 7 parse OK |

**RSA distribution (after fix):**
```
Buried (RSA < 0.2):   378/797 (47.4%)  — core + membrane + interfaces
Partial (0.2 - 0.5):  223/797 (28.0%)  — partially exposed
Exposed (> 0.5):      196/797 (24.6%)  — surface residues
```
This distribution is biologically reasonable for a pentameric membrane protein with extensive subunit interfaces.

---
## 6. PDB Structure Mapping — How Genes Map to Structures

nAChR is a **pentameric** receptor with 16 different subunit genes. We don't have a structure for every subunit, so we use:
- **Direct mapping** for subunits with experimental structures
- **Homology mapping** for closely related subunits
- **AlphaFold** for subunits with no experimental structure

### 6.1 PDB Structures Used

| PDB ID | Receptor | Resolution | Method | Chains |
|--------|----------|-----------|--------|--------|
| **9DMG** | Muscle α1β1δε | 2.05 Å | Cryo-EM | A/C=α1, B=ε, D=δ, E=β1 |
| **7EKT** | α7 closed | 3.20 Å | Cryo-EM | A-E=α7 (homopentamer) |
| **7KOX** | α7 open | 3.60 Å | Cryo-EM | A-E=α7 (activated) |
| **6CNJ** | α4β2 neuronal | 3.30 Å | Cryo-EM | A/C/E=α4, B/D=β2 (+ FG: β2, H: β2, I/J/K: α4) |
| **6PV7** | α3β4 neuronal | 2.80 Å | Cryo-EM | A/C=α3, B/D/E=β4 |
| **AF-Q9UGM1** | α9 | — | AlphaFold v6 | A=α9 (monomer) |
| **AF-Q13002** | α10 | — | AlphaFold v6 | A=α10 (monomer) |

### 6.2 Gene → PDB Mapping

https://www.rcsb.org/structure/9DMG

| Gene | PDB | Chain | Type | Notes |
|------|-----|-------|------|-------|
| CHRNA1 | 9DMG | A | Direct | Muscle α1 |
| CHRNB1 | 9DMG | E | Direct | Muscle β1 |
| CHRND | 9DMG | D | Direct | Muscle δ |
| CHRNE | 9DMG | B | Direct | Muscle ε (adult) |
| CHRNG | 9DMG | B | Homology→CHRNE | Fetal γ ≈ adult ε |
| CHRNA7 | 7EKT | A | Direct | α7 closed state |
| CHRNA4 | 6CNJ | A | Direct | α4β2 — principal α subunit |
| CHRNB2 | 6CNJ | B | Direct | α4β2 — complementary β subunit |
| CHRNA2 | 6CNJ | A | Homology→CHRNA4 | α2 ≈ α4 (~60% identity) |
| CHRNA6 | 6CNJ | A | Homology→CHRNA4 | α6 ≈ α4 |
| CHRNA3 | 6PV7 | A | Direct | α3β4 — principal α subunit |
| CHRNB4 | 6PV7 | B | Direct | α3β4 — complementary β subunit |
| CHRNA5 | 6PV7 | A | Homology→CHRNA3 | α5 ≈ α3 |
| CHRNB3 | 6PV7 | B | Homology→CHRNB4 | β3 ≈ β4 |
| CHRNA9 | AF-Q9UGM1 | A | AlphaFold | No experimental structure |
| CHRNA10 | AF-Q13002 | A | AlphaFold | No experimental structure |

### 6.3 The Alignment Problem

For each variant, we need to know: "which residue in the PDB structure corresponds to position X in UniProt?"

This is solved by **sequence alignment**:
1. Load the UniProt reference sequence for the gene (e.g., CHRNA7 from `data/raw/reference_sequences/human/CHRNA7.fasta`)
2. Extract the PDB chain sequence from the structure
3. Align using BioPython's `PairwiseAligner` with BLOSUM62 matrix
4. Build a mapping: `{uniprot_position: pdb_residue_id}`

**For direct-mapped genes:** Align the correct reference sequence to the correct chain.

**For homology-mapped genes:** Align the **homolog's** reference sequence (e.g., CHRNA2 reference to CHRNA4 chain). This works because the proteins are structurally similar, but sequence differences (~40% for distant homologs) cause alignment gaps → unmappable positions.

**For AlphaFold genes:** Align the gene's reference sequence to the AlphaFold structure chain. AlphaFold numbering typically matches UniProt exactly (1:1), so alignment is near-perfect.

---
## 7. Comparison with VEP-ENaC

| Aspect | VEP-ENaC | VEP-nAChR2 |
|--------|----------|------------|
| **PDB count** | 2 (6BQN, 6WTH) merged into one | 5 experimental + 2 AlphaFold = 7 |
| **Genes** | 4 (SCNN1A/B/G/D) | 16 (CHRNA1-7,9,10, CHRNB1-4, CHRND/E/G) |
| **Chains in PDB** | 3 (trimer) | 5 (pentamer) |
| **DSSP generation** | `mkdssp` via conda | `freesasa` + CIF for SS |
| **ASA algorithm** | mkdssp (Shrake-Rupley) | freesasa (Shrake-Rupley) |
| **SS assignment** | mkdssp (H-bond patterns) | mmCIF annotations (author-assigned) |
| **Structural features** | 5 (RSA, CB-density, B-factor, DSSP, unmappable) | 14 (5 core + 7 nAChR-specific + 2 placeholder) |
| **Conformational features** | None | 5 (CHRNA7 only, 1 active) |
| **Structural importance (ablation)** | **#1 most important** | Previously dead last (expected to improve after fix) |

**Key difference:** ENaC is a trimeric channel with 4 genes — simpler structure, better coverage. nAChR is pentameric with 16 genes — more complex, more homology gaps, more opportunities for alignment issues.

---
## 8. Remaining Issues & Future Work

### 8.1 Known Bugs

1. **structural_nachr skips alignment:** `_compute_pore_distance`, `_compute_interface_features`, and `_compute_min_distance` use **raw UniProt positions as PDB residue keys** instead of going through the alignment map. This works for modern well-annotated structures (where PDB numbering = UniProt numbering) but fails silently for others.

2. **subunit_burial is always 0:** Placeholder feature — requires separate monomeric and pentameric SASA calculations.

3. **Conformational features mostly placeholder:** Only CA RMSD is computed. Delta-RSA and delta-B-factor need real ASA from both states, which now works after the DSSP fix.

4. **CHRNA4 alignment:** 46% fill rate suggests the UniProt→PDB alignment for CHRNA4 (6CNJ chain A) has issues. The 6CNJ structure has multiple α4 chains (A, C, E, I, J, K) — chain A might not be the best choice for all variants.

5. **AlphaFold B-factors:** Stored as pLDDT confidence scores (0-100), not real thermal parameters. Models may misinterpret high-confidence predictions (pLDDT > 90) as "rigid" when they're not.

### 8.2 Planned Improvements

1. **Share alignment maps** between StructuralExtractor and StructuralNachrExtractor
2. **Compute full conformational delta features** for CHRNA7 now that ASA works
3. **Add mouse/rat reference sequences** for species-aware structural features
4. **Use species-matched PDBs** where available (e.g., mouse nAChR structures)
5. **Fix CHRNA4 alignment** — try alternate chains in 6CNJ

---
## 9. Key Abbreviations & Glossary

| Abbreviation | Full Form | Context |
|-------------|-----------|--------|
| **ASA** | Accessible Surface Area | Solvent-accessible area of a residue (Å²) |
| **RSA** | Relative Solvent Accessibility | ASA / MAX_ASA — normalized to [0,1] |
| **SASA** | Solvent Accessible Surface Area | Same as ASA |
| **DSSP** | Dictionary of Secondary Structure of Proteins | Program by Kabsch & Sander (1983) that computes SS + ASA from PDB coordinates |
| **mkdssp** | Make DSSP | The executable that runs the DSSP algorithm |
| **freesasa** | Free SASA | Open-source Python implementation of Shrake-Rupley SASA calculation |
| **PDB** | Protein Data Bank | Repository of 3D macromolecular structures (rcsb.org) |
| **mmCIF** | Macromolecular Crystallographic Information File | Modern replacement for PDB file format |
| **Cβ** | Carbon-beta | The beta carbon atom in an amino acid side chain |
| **CA / Cα** | Carbon-alpha | The alpha carbon atom — backbone reference point |
| **B-factor** | Temperature factor / Debye-Waller factor | Measure of atomic displacement (flexibility) in Å² |
| **KDTree** | K-Dimensional Tree | Spatial data structure for fast nearest-neighbor searches |
| **OPM** | Orientations of Proteins in Membranes | Database of membrane protein spatial positioning |
| **TM / TMD** | Transmembrane (Domain) | Protein region spanning the lipid bilayer |
| **ECD** | Extracellular Domain | N-terminal domain outside the cell (~210 residues in nAChR) |
| **ICD** | Intracellular Domain | Loop between TM3 and TM4 inside the cell |
| **pLDDT** | Predicted Local Distance Difference Test | AlphaFold's per-residue confidence metric (0-100) |
| **GOF** | Gain of Function | Mutation increases receptor activity |
| **LOF** | Loss of Function | Mutation decreases or abolishes receptor activity |
| **NNE** | No Net Effect | Mutation doesn't change receptor function |
| **Shrake-Rupley** | — | Algorithm for computing SASA by rolling a probe sphere over atomic spheres |
| **BLOSUM62** | BLOcks SUbstitution Matrix | Amino acid substitution scoring matrix used for sequence alignment |